In [1]:
import pandas as pd
import psycopg2

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "olist_bi",
    "user": "postgres",
    "password": "Akshat@2027"
}

conn = psycopg2.connect(**DB_CONFIG)

query = """
WITH review_data AS (
    SELECT
        r.review_id,
        r.order_id,
        r.review_score,
        r.review_comment_title,
        r.review_comment_message,
        o.order_purchase_timestamp,
        oi.product_id,
        oi.seller_id,
        p.product_category_name,
        COALESCE(
            ct.product_category_name_english,
            p.product_category_name
        ) AS category,
        ROW_NUMBER() OVER (
            PARTITION BY r.review_id
            ORDER BY oi.order_item_id
        ) AS rn
    FROM reviews r
    JOIN orders o
        ON r.order_id = o.order_id
    JOIN order_items oi
        ON r.order_id = oi.order_id
    JOIN products p
        ON oi.product_id = p.product_id
    LEFT JOIN category_translation ct
        ON p.product_category_name = ct.product_category_name
    WHERE r.review_comment_message IS NOT NULL
      AND TRIM(r.review_comment_message) <> ''
)

SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    order_purchase_timestamp,
    product_id,
    seller_id,
    product_category_name,
    category
FROM review_data
WHERE rn = 1;
"""

df = pd.read_sql_query(query, conn)

conn.close()

print("Rows:", len(df))
df.head()

Rows: 40155


/var/folders/nc/hvlqvddd2xj11jl0l69nx3580000gn/T/ipykernel_83589/1106900349.py:62: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


,review_id,order_id,review_score,review_comment_title,review_comment_message,order_purchase_timestamp,product_id,seller_id,product_category_name,category
0,00020c7512a52e92212f12d3e37513c0,e28abf2eb2f1fbcbdc2dd0cd9a561671,5,Entrega rápida!,A entrega foi super rápida e o pendente é lind...,2018-04-17 21:54:15,a39cc58c1b5926b6f9f378daa89f1315,1025f0e2d44d7041d6cf58b6550e0bfa,casa_construcao,home_construction
1,00046a69550325aea5fb89f65c7387f2,9fbda7367628952bc36c3512c46d887b,5,SUPER RECOMENDO,"GOSTEI DA CAPINHA DE CELULAR, VEIO COMO EU ESP...",2018-06-22 13:33:54,b5b2072f757c9317caa9c32c3bbf393d,2138ccb85b11a4ec1e37afbd1c8eda1f,telefonia,telephony
2,0005534973388c830bb858cfba83b17b,a589caa6892ceacc6bbf2f8cc30a8ad4,5,None,otimo produto. prazo cumprido. sabor tambem mu...,2017-08-15 17:32:48,da93657c402ea46729003c282352d727,c3cfdc648177fdbbbb35635a37472c53,esporte_lazer,sports_leisure
3,00055e36e9608fe969231e551983a69c,f5fea26ab547eec920e6f8ecdc5c37e4,5,None,O produto foi entregue muito antes do esperado...,2017-11-20 19:38:16,f02eae5fd95b59b1ec82397e1ef7ccf1,0ad80de75c811326389cfb1daba7d45e,brinquedos,toys
4,0005949d4c047d64863a6874338139ba,640883769bc5da4a077725646dc9995a,5,None,"Bom eu já sabia que era,mas é muito bonito.rec...",2017-06-05 18:20:04,c7c4e24200f7071aecf4a052fadb9473,0adac9fbd9a2b63cccaac4f8756c1ca8,utilidades_domesticas,housewares


In [2]:
print("Shape:", df.shape)
print("\nMissing values:")
print(df.isna().sum())

print("\nReview score distribution:")
print(df["review_score"].value_counts().sort_index())

print("\nCategories:")
print(df["category"].nunique())

print("\nSample reviews:")
display(
    df[
        ["review_score", "category", "review_comment_message"]
    ].sample(10, random_state=42)
)

Shape: (40155, 10)

Missing values:
review_id                       0
order_id                        0
review_score                    0
review_comment_title        30427
review_comment_message          0
order_purchase_timestamp        0
product_id                      0
seller_id                       0
product_category_name         605
category                      605
dtype: int64

Review score distribution:
review_score
1     8243
2     2077
3     3504
4     5927
5    20404
Name: count, dtype: int64

Categories:
73

Sample reviews:


,review_score,category,review_comment_message
15353,2,health_beauty,Muita demora na entrega.
31395,1,bed_bath_table,Comprei 2 e recebi apenas 1.\r\n\r\nNa nota fi...
33463,4,bed_bath_table,Produto de boa qualidade.\r\n
18967,1,telephony,"Comprei um carregador veicular da Belkin, poré..."
3935,5,auto,satisfeito com a compra
34421,4,sports_leisure,Só não veio na embalagem a mg de cada capsula....
21690,5,auto,Bom produto e lannister como sempre entregando...
17811,3,electronics,"Comprei um controle remoto Samsung, e assim ve..."
1278,2,furniture_decor,Não foi entregue
3942,5,signaling_and_security,Produto chegou dentro do prazo. Veio conforme ...


In [3]:
df["review_length"] = df["review_comment_message"].str.len()

print(df["review_length"].describe())

print("\nLongest reviews:")
display(
    df[
        [
            "review_score",
            "category",
            "review_comment_message",
            "review_length"
        ]
    ]
    .sort_values("review_length", ascending=False)
    .head(10)
)

count    40155.000000
mean        68.217507
std         53.606514
min          1.000000
25%         28.000000
50%         53.000000
75%         94.000000
max        208.000000
Name: review_length, dtype: float64

Longest reviews:


,review_score,category,review_comment_message,review_length
2794,1,bed_bath_table,"NÃO RECEBI O PRODUTO, O PRODUTO CONSTA COMO EN...",208
9429,4,luggage_accessories,Adquiri 2 mochilas pelo site lannister de e pa...,207
31274,1,None,Comprei o produto confiando nas lojas lanniste...,206
31960,1,cool_stuff,fiz essa compra pelo baratheon comprei um conj...,205
16597,1,sports_leisure,A EMPRESA FAZ PROPAGANDA ENGANOSA EM SEU SITE....,204
8582,4,computers_accessories,Comprei dois itens semelhantes (cartuchos de t...,204
14354,1,bed_bath_table,assim que fiz o pedido observei que tinha pedi...,204
6639,5,auto,"Parabéns ,o produto tira rico da empresa targa...",204
8101,1,cool_stuff,A targaryen não é de confiança não entregou a ...,204
36269,1,construction_tools_construction,Recebi a mercadoria errada.\r\nComprei o chuve...,204


In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Model loaded!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded!


In [5]:
sample_texts = df["review_comment_message"].sample(
    1000,
    random_state=42
).tolist()

sample_embeddings = model.encode(
    sample_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

print("Shape:", sample_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Shape: (1000, 384)


In [6]:
import numpy as np

texts = df["review_comment_message"].tolist()

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

embeddings = np.asarray(embeddings, dtype="float32")

print("Embedding shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Batches:   0%|          | 0/1255 [00:00<?, ?it/s]

Embedding shape: (40155, 384)
Embedding dtype: float32


In [7]:
print("Number of embeddings:", len(embeddings))
print("Embedding dimensions:", embeddings.shape[1])
print("First 10 values:")
print(embeddings[0][:10])

print("\nL2 norm of first embedding:")
print(np.linalg.norm(embeddings[0]))

Number of embeddings: 40155
Embedding dimensions: 384
First 10 values:
[-0.02326695  0.04753207 -0.04808553 -0.05042358 -0.00837246 -0.06394499
  0.05982044 -0.0034164  -0.00142585 -0.03807891]

L2 norm of first embedding:
1.0


In [8]:
%pip install qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 3.4 MB/s  0:00:03 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [qdrant-client]0m [h2]
Note: you may need to restart the kernel to use updated packages.


In [9]:
from qdrant_client import QdrantClient

client = QdrantClient(":memory:")

print("Qdrant initialized!")

Qdrant initialized!


In [10]:
from qdrant_client.models import Distance, VectorParams

collection_name = "olist_reviews"

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(
        size=384,
        distance=Distance.COSINE
    )
)

print(f"Collection '{collection_name}' created!")

Collection 'olist_reviews' created!


In [11]:
from qdrant_client.models import PointStruct

points = []

for i, row in df.iterrows():
    points.append(
        PointStruct(
            id=i,
            vector=embeddings[i].tolist(),
            payload={
                "review_id": row["review_id"],
                "order_id": row["order_id"],
                "review_score": int(row["review_score"]),
                "review_text": row["review_comment_message"],
                "category": row["category"],
                "product_category_name": row["product_category_name"],
                "product_id": row["product_id"],
                "seller_id": row["seller_id"],
                "order_purchase_timestamp": str(
                    row["order_purchase_timestamp"]
                )
            }
        )
    )

print("Points prepared:", len(points))

Points prepared: 40155


In [12]:
from qdrant_client.models import PointStruct

BATCH_SIZE = 1000

for start in range(0, len(points), BATCH_SIZE):
    batch = points[start:start + BATCH_SIZE]

    client.upsert(
        collection_name=collection_name,
        points=batch
    )

    print(
        f"Uploaded {min(start + BATCH_SIZE, len(points)):,}"
        f" / {len(points):,}"
    )

print("Upload complete!")

Uploaded 1,000 / 40,155
Uploaded 2,000 / 40,155
Uploaded 3,000 / 40,155
Uploaded 4,000 / 40,155
Uploaded 5,000 / 40,155
Uploaded 6,000 / 40,155
Uploaded 7,000 / 40,155
Uploaded 8,000 / 40,155
Uploaded 9,000 / 40,155
Uploaded 10,000 / 40,155
Uploaded 11,000 / 40,155
Uploaded 12,000 / 40,155
Uploaded 13,000 / 40,155
Uploaded 14,000 / 40,155
Uploaded 15,000 / 40,155
Uploaded 16,000 / 40,155
Uploaded 17,000 / 40,155
Uploaded 18,000 / 40,155
Uploaded 19,000 / 40,155
Uploaded 20,000 / 40,155


/var/folders/nc/hvlqvddd2xj11jl0l69nx3580000gn/T/ipykernel_83589/1711743457.py:8: UserWarning: Local mode is not recommended for collections with more than 20,000 points. Current collection contains 21000 points. Consider using Qdrant in Docker or Qdrant Cloud for better performance with large datasets.
  client.upsert(


Uploaded 21,000 / 40,155
Uploaded 22,000 / 40,155
Uploaded 23,000 / 40,155
Uploaded 24,000 / 40,155
Uploaded 25,000 / 40,155
Uploaded 26,000 / 40,155
Uploaded 27,000 / 40,155
Uploaded 28,000 / 40,155
Uploaded 29,000 / 40,155
Uploaded 30,000 / 40,155
Uploaded 31,000 / 40,155
Uploaded 32,000 / 40,155
Uploaded 33,000 / 40,155
Uploaded 34,000 / 40,155
Uploaded 35,000 / 40,155
Uploaded 36,000 / 40,155
Uploaded 37,000 / 40,155
Uploaded 38,000 / 40,155
Uploaded 39,000 / 40,155
Uploaded 40,000 / 40,155
Uploaded 40,155 / 40,155
Upload complete!


In [13]:
info = client.get_collection(collection_name)

print("Vectors stored:", info.points_count)

Vectors stored: 40155


In [14]:
query = "customers complaining about damaged or defective furniture"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

results = client.query_points(
    collection_name=collection_name,
    query=query_embedding.tolist(),
    limit=10
)

for i, result in enumerate(results.points, 1):
    print(f"\n--- Result {i} ---")
    print("Score:", round(result.score, 4))
    print("Rating:", result.payload["review_score"])
    print("Category:", result.payload["category"])
    print("Review:", result.payload["review_text"])


--- Result 1 ---
Score: 0.5015
Rating: 1
Category: bed_bath_table
Review: capa de sofa muito horrivel pessima qualidade 

--- Result 2 ---
Score: 0.4808
Rating: 1
Category: cool_stuff
Review: vou registrar queixa no reclameaqui porque, apesar de informar que, me enviaram produto para tensao errada (pedi 110V e mandaram 220V) apresentei reclamaçao e nao me deram satisfaçao

--- Result 3 ---
Score: 0.4778
Rating: 4
Category: stationery
Review: tudo certo com o produto. Mas a loja exagerou na embalagem. Muito plástico e papelão. Cade a preocupação com a sustentabilidade?

--- Result 4 ---
Score: 0.4754
Rating: 1
Category: housewares
Review: Gostaria de saber a quem e como reclamar para receber o produto correto.
Grata.

--- Result 5 ---
Score: 0.4665
Rating: 1
Category: computers_accessories
Review: Produto danificado e não aceita devolução ja estou abrindo reclamação no reclame aqui e se não obtiver resultado abrirei para lannister também .

--- Result 6 ---
Score: 0.4637
Rating: 3
Cate

In [15]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

query = "customers complaining about damaged or defective furniture"

query_embedding = model.encode(
    query,
    normalize_embeddings=True
)

results = client.query_points(
    collection_name=collection_name,
    query=query_embedding.tolist(),
    query_filter=Filter(
        must=[
            FieldCondition(
                key="category",
                match=MatchValue(value="furniture_decor")
            )
        ]
    ),
    limit=10
)

for i, result in enumerate(results.points, 1):
    print(f"\n--- Result {i} ---")
    print("Score:", round(result.score, 4))
    print("Rating:", result.payload["review_score"])
    print("Review:", result.payload["review_text"])


--- Result 1 ---
Score: 0.4293
Rating: 1
Review: Fiz a compra de duas cortinas e só me enviaram uma abri uma reclamação e ate agora nem me responderam e nem deram satisfação é eu consumidora que estou no prejuízo.

--- Result 2 ---
Score: 0.4204
Rating: 1
Review: Produto com defeito 
Preciso trocar


--- Result 3 ---
Score: 0.4191
Rating: 1
Review: O produto veio com defeito, a cortina veio com a embalagem violada e o produto estava danificado, queimado.
Fiz a reclamação direto com a loja e eles ficaram de me retornar mas não retornaram até hoj

--- Result 4 ---
Score: 0.4164
Rating: 1
Review: Antes de recomendar a loja preciso ver se a loja irá trocar o produto que veio com defeito. Para mim minha compra não foi finalizada ainda.

--- Result 5 ---
Score: 0.4061
Rating: 1
Review: A cadeira decorativa veio c/ pé trocado, c/ solda aparente, acabamento mal feito. estofados vieram s/ nenhuma proteção (plástico) numa caixa toda remendada, s/ manual e medidas não conferem.

--- Result 6 ---

In [16]:
evaluation_queries = [
    {
        "query": "customers complaining about damaged or defective products",
        "category": "furniture_decor"
    },
    {
        "query": "customers complaining about late delivery",
        "category": "furniture_decor"
    },
    {
        "query": "customers complaining about missing items",
        "category": "furniture_decor"
    },
    {
        "query": "customers complaining about poor product quality",
        "category": "furniture_decor"
    },
    {
        "query": "customers complaining about sellers not responding",
        "category": "furniture_decor"
    },
    {
        "query": "customers asking for refunds or replacements",
        "category": "furniture_decor"
    },
    {
        "query": "customers receiving the wrong product",
        "category": "furniture_decor"
    },
    {
        "query": "customers complaining about poor packaging",
        "category": "furniture_decor"
    },
    {
        "query": "customers saying their order was not delivered",
        "category": "furniture_decor"
    },
    {
        "query": "customers complaining about product dimensions or size",
        "category": "furniture_decor"
    }
]

print("Evaluation queries:", len(evaluation_queries))

Evaluation queries: 10


In [17]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

evaluation_results = []

for item in evaluation_queries:

    query = item["query"]
    category = item["category"]

    query_embedding = model.encode(
        query,
        normalize_embeddings=True
    )

    results = client.query_points(
        collection_name=collection_name,
        query=query_embedding.tolist(),
        query_filter=Filter(
            must=[
                FieldCondition(
                    key="category",
                    match=MatchValue(value=category)
                )
            ]
        ),
        limit=5
    )

    for rank, result in enumerate(results.points, 1):
        evaluation_results.append({
            "query": query,
            "category": category,
            "rank": rank,
            "score": result.score,
            "review_id": result.payload["review_id"],
            "review_score": result.payload["review_score"],
            "review_text": result.payload["review_text"]
        })

eval_df = pd.DataFrame(evaluation_results)

print("Retrieved results:", len(eval_df))

Retrieved results: 50


In [18]:
display(eval_df)

,query,category,rank,score,review_id,review_score,review_text
0,customers complaining about damaged or defecti...,furniture_decor,1,0.600092,f52c8058817a332008a3c374318bf70c,4,Insatisfação com a descrição do produto.
1,customers complaining about damaged or defecti...,furniture_decor,2,0.573020,ab2a7a5b86afdac6192523b81cf562b6,1,Produto com defeito \r\nPreciso trocar\r\n
2,customers complaining about damaged or defecti...,furniture_decor,3,0.564361,09e1f7e3288ae0f68dda1d8f69f37bb2,1,Antes de recomendar a loja preciso ver se a lo...
3,customers complaining about damaged or defecti...,furniture_decor,4,0.550699,eada60766110d984171b1a231c1e3600,1,produto com defeito
4,customers complaining about damaged or defecti...,furniture_decor,5,0.545876,5b2602b06d6360ecf96806983c564cb8,1,"O produto veio com defeito, a cortina veio com..."
5,customers complaining about late delivery,furniture_decor,1,0.649404,dd6e0f7168d75cd78769333f7630a036,3,O atraso na entrega prejudicou a obra
6,customers complaining about late delivery,furniture_decor,2,0.636680,1e39301e37ad6658793dfe56c6e0c077,1,produto atrasado sem resposta da loja
7,customers complaining about late delivery,furniture_decor,3,0.625134,574ff7b2058b5fbb967b36ccf690c472,3,"Sou cliente stark há anos, mas o motivo do meu..."
8,customers complaining about late delivery,furniture_decor,4,0.624970,06ca23fe7c7c38193969c38098aca1bd,2,A loja me ligou informando que a transportador...
9,customers complaining about late delivery,furniture_decor,5,0.617550,98c303ce6a5deb174c7a4b76c882d940,1,Produto comprado e não entregue no prazo. O fo...


In [24]:
print("QUERY:", query)

QUERY: customers complaining about product dimensions or size


In [26]:
for query in eval_df["query"].unique():

    print("\n" + "=" * 80)
    print("QUERY:", query)
    print("=" * 80)

    subset = eval_df[eval_df["query"] == query].sort_values("rank")

    for _, row in subset.iterrows():
        print(
            f"{int(row['rank'])}. "
            f"score={row['score']:.4f} | "
            f"rating={int(row['review_score'])} | "
            f"{row['review_text'][:180]}"
        )


QUERY: customers complaining about damaged or defective products
1. score=0.6001 | rating=4 | Insatisfação com a descrição do produto.
2. score=0.5730 | rating=1 | Produto com defeito 
Preciso trocar

3. score=0.5644 | rating=1 | Antes de recomendar a loja preciso ver se a loja irá trocar o produto que veio com defeito. Para mim minha compra não foi finalizada ainda.
4. score=0.5507 | rating=1 | produto com defeito
5. score=0.5459 | rating=1 | O produto veio com defeito, a cortina veio com a embalagem violada e o produto estava danificado, queimado.
Fiz a reclamação direto com a loja e eles ficaram de me retornar mas nã

QUERY: customers complaining about late delivery
1. score=0.6494 | rating=3 | O atraso na entrega prejudicou a obra
2. score=0.6367 | rating=1 | produto atrasado sem resposta da loja 
3. score=0.6251 | rating=3 | Sou cliente stark há anos, mas o motivo do meu descontentamento é porque pela segunda vez me enviaram um produto danificado e esta entrega está atrasada. Ao 

In [28]:
evaluation_summary = (
    eval_df
    .groupby("query")
    .agg(
        relevant_ranks=(
            "rank",
            lambda x: ",".join(
                map(str, x[eval_df.loc[x.index, "relevant"] == 1])
            )
        ),
        precision_at_5=("relevant", "mean")
    )
    .reset_index()
)

evaluation_summary

,query,relevant_ranks,precision_at_5
0,customers asking for refunds or replacements,,0.0
1,customers complaining about damaged or defecti...,"2,3,4,5",0.8
2,customers complaining about late delivery,"1,2,3,4,5",1.0
3,customers complaining about missing items,"2,3,4",0.6
4,customers complaining about poor packaging,,0.0
5,customers complaining about poor product quality,"2,3,4",0.6
6,customers complaining about product dimensions...,"2,3",0.4
7,customers complaining about sellers not respon...,"1,5",0.4
8,customers receiving the wrong product,"1,3",0.4
9,customers saying their order was not delivered,"1,2,3,4,5",1.0


In [29]:
for query, group in eval_df.groupby("query", sort=False):

    relevant_ranks = group.loc[
        group["relevant"] == 1, "rank"
    ].tolist()

    precision_at_5 = group["relevant"].mean()

    if relevant_ranks:
        ranks_str = ",".join(map(str, relevant_ranks))
    else:
        ranks_str = "none clearly relevant"

    print(f"{query:<65} | {ranks_str:<20} | {precision_at_5:.2f}")

customers complaining about damaged or defective products         | 2,3,4,5              | 0.80
customers complaining about late delivery                         | 1,2,3,4,5            | 1.00
customers complaining about missing items                         | 2,3,4                | 0.60
customers complaining about poor product quality                  | 2,3,4                | 0.60
customers complaining about sellers not responding                | 1,5                  | 0.40
customers asking for refunds or replacements                      | none clearly relevant | 0.00
customers receiving the wrong product                             | 1,3                  | 0.40
customers complaining about poor packaging                        | none clearly relevant | 0.00
customers saying their order was not delivered                    | 1,2,3,4,5            | 1.00
customers complaining about product dimensions or size            | 2,3                  | 0.40


In [30]:
precision_at_5 = eval_df["relevant"].mean()

# MRR
reciprocal_ranks = []

for query, group in eval_df.groupby("query", sort=False):
    relevant = group.loc[group["relevant"] == 1, "rank"]

    if len(relevant) > 0:
        reciprocal_ranks.append(1 / relevant.min())
    else:
        reciprocal_ranks.append(0)

mrr = np.mean(reciprocal_ranks)

print(f"\nMean Precision@5: {precision_at_5:.2f}")
print(f"MRR: {mrr:.2f}")


Mean Precision@5: 0.52
MRR: 0.60


# Add TF-IDF retrieval

In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

In [32]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(
    df["review_comment_message"]
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)

TF-IDF matrix shape: (40155, 37416)


In [33]:
def tfidf_search(query, category=None, top_k=5):

    query_vector = tfidf.transform([query])

    scores = cosine_similarity(
        query_vector,
        tfidf_matrix
    ).flatten()

    # Category filter
    if category:
        valid_indices = df.index[
            df["category"] == category
        ]

        scores_filtered = scores[valid_indices]

        top_positions = np.argsort(
            scores_filtered
        )[::-1][:top_k]

        top_indices = valid_indices[top_positions]

    else:
        top_indices = np.argsort(scores)[::-1][:top_k]

    results = df.loc[top_indices].copy()

    results["tfidf_score"] = scores[top_indices]

    return results[
        [
            "review_id",
            "review_score",
            "category",
            "review_comment_message",
            "tfidf_score"
        ]
    ]

In [34]:
results = tfidf_search(
    "customers complaining about poor packaging",
    category="furniture_decor",
    top_k=5
)

results

,review_id,review_score,category,review_comment_message,tfidf_score
40134,ffd60a4fcda4d97484df9ff62d65303b,5,furniture_decor,Chegou tudo direitinho bem embalado para não q...,0.0
13093,5373204e340f16043e14faf2eecd7d11,5,furniture_decor,"Muito bom, Tenho pena que vocês não enviam par...",0.0
13159,53d5d2c0713c5c75d5dc016a2eb12546,5,furniture_decor,Recebi antes do esperado.,0.0
13155,53d28d3700e6e8caaf605fee4818d167,5,furniture_decor,"Excelente produto, acdorei! Compraria novamente!",0.0
13145,53bdd47e56c31c07e0c8b7c5a412a0b3,1,furniture_decor,"Ainda não recebi o produto, e considerando a p...",0.0


In [35]:
for _, row in results.iterrows():
    print(
        f"\nScore: {row['tfidf_score']:.4f}"
        f"\nRating: {row['review_score']}"
        f"\nReview: {row['review_comment_message']}"
    )


Score: 0.0000
Rating: 5
Review: Chegou tudo direitinho bem embalado para não quebrar.
Muito obrigada

Score: 0.0000
Rating: 5
Review: Muito bom, Tenho pena que vocês não enviam para o Norte do Brasil.

Score: 0.0000
Rating: 5
Review: Recebi antes do esperado. 

Score: 0.0000
Rating: 5
Review: Excelente produto, acdorei! Compraria novamente!

Score: 0.0000
Rating: 1
Review: Ainda não recebi o produto, e considerando a paralisação de caminhoneiros, já era pra ter sido entregue. 


In [36]:
tfidf_eval_results = []

for query in eval_df["query"].unique():

    category = eval_df.loc[
        eval_df["query"] == query,
        "category"
    ].iloc[0]

    results = tfidf_search(
        query,
        category=category,
        top_k=5
    )

    labels = eval_df[
        eval_df["query"] == query
    ][["review_id", "relevant"]]

    results = results.merge(
        labels,
        on="review_id",
        how="left"
    )

    results["relevant"] = results["relevant"].fillna(0)

    results["query"] = query

    tfidf_eval_results.append(results)

tfidf_eval_df = pd.concat(
    tfidf_eval_results,
    ignore_index=True
)

In [37]:
precision_at_5 = tfidf_eval_df["relevant"].mean()

reciprocal_ranks = []

for query, group in tfidf_eval_df.groupby("query", sort=False):

    relevant = group.loc[
        group["relevant"] == 1
    ]

    if len(relevant) > 0:
        first_relevant_rank = relevant.index[0]
        reciprocal_ranks.append(
            1 / (group.index.get_loc(first_relevant_rank) + 1)
        )
    else:
        reciprocal_ranks.append(0)

mrr = np.mean(reciprocal_ranks)

print(f"TF-IDF Mean Precision@5: {precision_at_5:.2f}")
print(f"TF-IDF MRR: {mrr:.2f}")

TF-IDF Mean Precision@5: 0.00
TF-IDF MRR: 0.00


In [40]:
print("qdrant:", "qdrant" in globals())
print("model:", "model" in globals())
print("df:", "df" in globals())
print("tfidf:", "tfidf" in globals())
print("eval_df:", "eval_df" in globals())

qdrant: False
model: True
df: True
tfidf: True
eval_df: True


In [41]:
print("embeddings:", "embeddings" in globals())

if "embeddings" in globals():
    print(embeddings.shape)

embeddings: True
(40155, 384)


In [42]:
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct,
    Filter,
    FieldCondition,
    MatchValue
)

qdrant = QdrantClient(":memory:")

qdrant.create_collection(
    collection_name="olist_reviews",
    vectors_config=VectorParams(
        size=embeddings.shape[1],
        distance=Distance.COSINE
    )
)

True

In [43]:
points = []

for idx, row in df.iterrows():

    points.append(
        PointStruct(
            id=int(idx),
            vector=embeddings[idx].tolist(),
            payload={
                "review_id": row["review_id"],
                "order_id": row["order_id"],
                "review_score": int(row["review_score"]),
                "review_text": row["review_comment_message"],
                "category": row["category"],
                "product_category_name": row["product_category_name"],
                "product_id": row["product_id"],
                "seller_id": row["seller_id"],
                "order_purchase_timestamp": str(
                    row["order_purchase_timestamp"]
                )
            }
        )
    )

In [44]:
batch_size = 1000

for i in range(0, len(points), batch_size):

    qdrant.upsert(
        collection_name="olist_reviews",
        points=points[i:i + batch_size]
    )

print(
    "Qdrant count:",
    qdrant.count(
        collection_name="olist_reviews"
    ).count
)

Qdrant count: 40155


In [50]:
candidate_pool = []

for query in eval_df["query"].unique():

    category = eval_df.loc[
        eval_df["query"] == query,
        "category"
    ].iloc[0]

    # Semantic candidates
    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    semantic_response = qdrant.query_points(
        collection_name="olist_reviews",
        query=query_embedding,
        query_filter=Filter(
            must=[
                FieldCondition(
                    key="category",
                    match=MatchValue(value=category)
                )
            ]
        ),
        limit=5
    )

    semantic_results = semantic_response.points

    semantic_ids = [
        r.payload["review_id"]
        for r in semantic_results
    ]

    # TF-IDF candidates
    tfidf_results = tfidf_search(
        query,
        category=category,
        top_k=5
    )

    tfidf_ids = tfidf_results["review_id"].tolist()

    # Combine without duplicates
    candidate_ids = list(
        dict.fromkeys(
            semantic_ids + tfidf_ids
        )
    )

    for review_id in candidate_ids:

        row = df[
            df["review_id"] == review_id
        ].iloc[0]

        candidate_pool.append({
            "query": query,
            "category": category,
            "review_id": review_id,
            "review_score": row["review_score"],
            "review_text": row["review_comment_message"],
            "relevant": None
        })

candidate_pool_df = pd.DataFrame(candidate_pool)

print("Total candidates:", len(candidate_pool_df))

Total candidates: 100


In [51]:
print("Total candidates:", len(candidate_pool_df))

Total candidates: 100


In [52]:
# Copy existing semantic relevance labels
semantic_labels = eval_df[
    ["query", "review_id", "relevant"]
].copy()

semantic_labels = semantic_labels.rename(
    columns={"relevant": "semantic_relevant"}
)

candidate_pool_df = candidate_pool_df.merge(
    semantic_labels,
    on=["query", "review_id"],
    how="left"
)

# Identify which candidates are new
candidate_pool_df["needs_label"] = (
    candidate_pool_df["semantic_relevant"].isna()
)

print(
    "Already labeled:",
    (~candidate_pool_df["needs_label"]).sum()
)

print(
    "Need manual labeling:",
    candidate_pool_df["needs_label"].sum()
)

Already labeled: 50
Need manual labeling: 50


In [53]:
new_candidates = candidate_pool_df[
    candidate_pool_df["needs_label"]
][
    [
        "query",
        "category",
        "review_id",
        "review_score",
        "review_text"
    ]
].copy()

new_candidates.to_csv(
    "tfidf_candidates_to_label.csv",
    index=False
)

print("New candidates:", len(new_candidates))

New candidates: 50


In [55]:
new_candidates.to_csv(
    "tfidf_candidates_to_label.csv",
    index=False
)

## NOW :--

# Query
#  ↓
# Qdrant semantic retrieval
#  ↓
# Top-20 candidates
#  ↓
# Cross-encoder reranker
#  ↓
# Top-5

In [57]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
)

config.json:   0%|          | 0.00/891 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

sentencepiece.bpe.model: downloading bytes:           |  0.00B            

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [58]:
reranker

CrossEncoder(
  (0): Transformer({'transformer_task': 'sequence-classification', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'logits'}}, 'module_output_name': 'scores', 'architecture': 'XLMRobertaForSequenceClassification'})
)

In [59]:
query = "customers complaining about poor packaging"
category = "furniture_decor"

In [60]:
query_embedding = model.encode(
    [query],
    normalize_embeddings=True
)[0]

semantic_response = qdrant.query_points(
    collection_name="olist_reviews",
    query=query_embedding,
    query_filter=Filter(
        must=[
            FieldCondition(
                key="category",
                match=MatchValue(value=category)
            )
        ]
    ),
    limit=20
)

semantic_results = semantic_response.points

print("Retrieved:", len(semantic_results))

Retrieved: 20


In [61]:
pairs = [
    [
        query,
        result.payload["review_text"]
    ]
    for result in semantic_results
]

In [62]:
rerank_scores = reranker.predict(pairs)

In [63]:
reranked = sorted(
    zip(semantic_results, rerank_scores),
    key=lambda x: x[1],
    reverse=True
)

In [64]:
reranked_df = pd.DataFrame([
    {
        "rank": i + 1,
        "rerank_score": float(score),
        "review_id": result.payload["review_id"],
        "review_score": result.payload["review_score"],
        "review_text": result.payload["review_text"]
    }
    for i, (result, score) in enumerate(reranked[:5])
])

reranked_df

,rank,rerank_score,review_id,review_score,review_text
0,1,-1.748540,118cea1913e2f9d91f24066bf570fe20,1,"péssima embalagem , tive que colar ele todo, f..."
1,2,-1.769024,e728b7d132f8dfa2229cb6050e93a88f,1,"Recebi a compra pela metade, comprei 03 kits e..."
2,3,-2.829854,574ff7b2058b5fbb967b36ccf690c472,3,"Sou cliente stark há anos, mas o motivo do meu..."
3,4,-3.022887,f52c8058817a332008a3c374318bf70c,4,Insatisfação com a descrição do produto.
4,5,-3.836206,d69aa04a78d0651dba8163e736abecb5,1,Atendimento péssimo das lojas lannister tambem...


In [65]:
for _, row in reranked_df.iterrows():
    print(
        f"\nRank {row['rank']} | "
        f"Score: {row['rerank_score']:.4f} | "
        f"Rating: {row['review_score']}"
    )
    print(row["review_text"])


Rank 1 | Score: -1.7485 | Rating: 1
péssima embalagem , tive que colar ele todo, fique com preguiça pelo preço de mandar de volta, mas não indico pra ninguém a loja...

Rank 2 | Score: -1.7690 | Rating: 1
Recebi a compra pela metade, comprei 03 kits e só veio 01... péssima compra... já formalizei a reclamação a loja e ate o momento nada de retorno!

Rank 3 | Score: -2.8299 | Rating: 3
Sou cliente stark há anos, mas o motivo do meu descontentamento é porque pela segunda vez me enviaram um produto danificado e esta entrega está atrasada. Ao tentar entrar em contato com a Central 

Rank 4 | Score: -3.0229 | Rating: 4
Insatisfação com a descrição do produto.

Rank 5 | Score: -3.8362 | Rating: 1
Atendimento péssimo das lojas lannister tambem. Produto incompleto.nao merece nenhuma estrela.


In [66]:
reranker_eval = []

for query in eval_df["query"].unique():

    category = eval_df.loc[
        eval_df["query"] == query,
        "category"
    ].iloc[0]

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    response = qdrant.query_points(
        collection_name="olist_reviews",
        query=query_embedding,
        query_filter=Filter(
            must=[
                FieldCondition(
                    key="category",
                    match=MatchValue(value=category)
                )
            ]
        ),
        limit=20
    )

    candidates = response.points

    pairs = [
        [query, r.payload["review_text"]]
        for r in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )[:5]

    for rank, (result, score) in enumerate(ranked, 1):

        reranker_eval.append({
            "query": query,
            "category": category,
            "rank": rank,
            "review_id": result.payload["review_id"],
            "rerank_score": float(score),
            "review_score": result.payload["review_score"],
            "review_text": result.payload["review_text"]
        })

reranker_eval_df = pd.DataFrame(reranker_eval)

print("Results:", len(reranker_eval_df))

Results: 50


In [67]:
reranker_labeled = reranker_eval_df.merge(
    eval_df[["query", "review_id", "relevant"]],
    on=["query", "review_id"],
    how="left"
)

print(
    "Already labeled:",
    reranker_labeled["relevant"].notna().sum()
)

print(
    "Need manual labeling:",
    reranker_labeled["relevant"].isna().sum()
)

Already labeled: 17
Need manual labeling: 33


In [68]:
new_reranker_candidates = reranker_labeled[
    reranker_labeled["relevant"].isna()
][
    [
        "query",
        "category",
        "rank",
        "review_id",
        "rerank_score",
        "review_score",
        "review_text"
    ]
].copy()

print("Candidates to label:", len(new_reranker_candidates))

new_reranker_candidates.to_csv(
    "reranker_candidates_to_label.csv",
    index=False
)

Candidates to label: 33


In [70]:
new_labels = {
    # Damaged / defective: new ranks 1, 3, 4
    ("customers complaining about damaged or defective products", 1): 1,
    ("customers complaining about damaged or defective products", 3): 1,
    ("customers complaining about damaged or defective products", 4): 0,

    # Late delivery: new ranks 2, 3, 4
    ("customers complaining about late delivery", 2): 1,
    ("customers complaining about late delivery", 3): 1,
    ("customers complaining about late delivery", 4): 1,

    # Missing items: new ranks 1, 3, 4, 5
    ("customers complaining about missing items", 1): 1,
    ("customers complaining about missing items", 3): 1,
    ("customers complaining about missing items", 4): 0,
    ("customers complaining about missing items", 5): 0,

    # Poor product quality: new ranks 1, 4, 5
    ("customers complaining about poor product quality", 1): 1,
    ("customers complaining about poor product quality", 4): 1,
    ("customers complaining about poor product quality", 5): 0,

    # Sellers not responding: new ranks 3, 4, 5
    ("customers complaining about sellers not responding", 3): 1,
    ("customers complaining about sellers not responding", 4): 1,
    ("customers complaining about sellers not responding", 5): 0,

    # Refunds / replacements: new ranks 1-5
    ("customers asking for refunds or replacements", 1): 1,
    ("customers asking for refunds or replacements", 2): 1,
    ("customers asking for refunds or replacements", 3): 1,
    ("customers asking for refunds or replacements", 4): 0,
    ("customers asking for refunds or replacements", 5): 0,

    # Wrong product: new ranks 1, 2, 4, 5
    ("customers receiving the wrong product", 1): 1,
    ("customers receiving the wrong product", 2): 0,
    ("customers receiving the wrong product", 4): 0,
    ("customers receiving the wrong product", 5): 0,

    # Poor packaging: new ranks 1, 2, 3
    ("customers complaining about poor packaging", 1): 1,
    ("customers complaining about poor packaging", 2): 0,
    ("customers complaining about poor packaging", 3): 0,

    # Order not delivered: new ranks 1, 2, 3, 5
    ("customers saying their order was not delivered", 1): 1,
    ("customers saying their order was not delivered", 2): 1,
    ("customers saying their order was not delivered", 3): 1,
    ("customers saying their order was not delivered", 5): 1,

    # Dimensions / size: new rank 5
    ("customers complaining about product dimensions or size", 5): 0,
}

In [71]:
for idx, row in reranker_labeled.iterrows():

    if pd.isna(row["relevant"]):
        key = (row["query"], int(row["rank"]))

        if key not in new_labels:
            print("MISSING LABEL:", key)
            continue

        reranker_labeled.loc[idx, "relevant"] = new_labels[key]

reranker_labeled["relevant"] = (
    reranker_labeled["relevant"].astype(int)
)

print("Total rows:", len(reranker_labeled))
print("Total labeled:", reranker_labeled["relevant"].notna().sum())

Total rows: 50
Total labeled: 50


In [72]:
precision_at_5 = reranker_labeled["relevant"].mean()

reciprocal_ranks = []

for query, group in reranker_labeled.groupby("query", sort=False):

    relevant = group.loc[
        group["relevant"] == 1,
        "rank"
    ]

    if len(relevant) > 0:
        reciprocal_ranks.append(1 / relevant.min())
    else:
        reciprocal_ranks.append(0)

mrr = np.mean(reciprocal_ranks)

print(f"Reranker Precision@5: {precision_at_5:.2f}")
print(f"Reranker MRR: {mrr:.2f}")

Reranker Precision@5: 0.58
Reranker MRR: 0.90


In [73]:
query_metrics = []

for query, group in reranker_labeled.groupby("query", sort=False):

    relevant = group["relevant"].values

    precision = relevant.mean()

    relevant_ranks = group.loc[
        group["relevant"] == 1,
        "rank"
    ]

    if len(relevant_ranks) > 0:
        rr = 1 / relevant_ranks.min()
    else:
        rr = 0

    query_metrics.append({
        "query": query,
        "Precision@5": precision,
        "RR": rr
    })

query_metrics_df = pd.DataFrame(query_metrics)

query_metrics_df

,query,Precision@5,RR
0,customers complaining about damaged or defecti...,0.8,1.0
1,customers complaining about late delivery,1.0,1.0
2,customers complaining about missing items,0.4,1.0
3,customers complaining about poor product quality,0.6,1.0
4,customers complaining about sellers not respon...,0.6,0.5
5,customers asking for refunds or replacements,0.6,1.0
6,customers receiving the wrong product,0.2,1.0
7,customers complaining about poor packaging,0.2,1.0
8,customers saying their order was not delivered,1.0,1.0
9,customers complaining about product dimensions...,0.4,0.5


In [75]:
comparison = pd.DataFrame({
    "Retriever": [
        "Semantic Qdrant",
        "Qdrant + Cross-Encoder Reranker"
    ],
    "Precision@5": [
        0.52,
        0.58
    ],
    "MRR": [
        0.60,
        0.90
    ]
})

comparison

,Retriever,Precision@5,MRR
0,Semantic Qdrant,0.52,0.6
1,Qdrant + Cross-Encoder Reranker,0.58,0.9


In [76]:
def retrieve_reviews(query, category=None, candidate_k=20, top_k=5):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )[0]

    query_filter = None

    if category:
        query_filter = Filter(
            must=[
                FieldCondition(
                    key="category",
                    match=MatchValue(value=category)
                )
            ]
        )

    response = qdrant.query_points(
        collection_name="olist_reviews",
        query=query_embedding,
        query_filter=query_filter,
        limit=candidate_k
    )

    candidates = response.points

    pairs = [
        [query, result.payload["review_text"]]
        for result in candidates
    ]

    scores = reranker.predict(pairs)

    ranked = sorted(
        zip(candidates, scores),
        key=lambda x: x[1],
        reverse=True
    )[:top_k]

    results = []

    for rank, (result, score) in enumerate(ranked, 1):

        payload = result.payload

        results.append({
            "rank": rank,
            "rerank_score": float(score),
            "review_id": payload["review_id"],
            "review_score": payload["review_score"],
            "review_text": payload["review_text"],
            "category": payload["category"],
            "product_id": payload["product_id"],
            "seller_id": payload["seller_id"],
            "order_purchase_timestamp":
                payload["order_purchase_timestamp"]
        })

    return pd.DataFrame(results)

In [77]:
results = retrieve_reviews(
    "customers complaining about poor packaging",
    category="furniture_decor"
)

results

,rank,rerank_score,review_id,review_score,review_text,category,product_id,seller_id,order_purchase_timestamp
0,1,-1.748540,118cea1913e2f9d91f24066bf570fe20,1,"péssima embalagem , tive que colar ele todo, f...",furniture_decor,96ed3b4b62790bd962aac0b317109be4,cac4c8e7b1ca6252d8f20b2fc1a2e4af,2017-05-18 15:23:08
1,2,-1.769024,e728b7d132f8dfa2229cb6050e93a88f,1,"Recebi a compra pela metade, comprei 03 kits e...",furniture_decor,e4e81849dd9ccdefe1e7398792a6a4b4,77530e9772f57a62c906e1c21538ab82,2017-05-11 10:19:00
2,3,-2.829854,574ff7b2058b5fbb967b36ccf690c472,3,"Sou cliente stark há anos, mas o motivo do meu...",furniture_decor,3c5ba90e09547db90868e803d139de30,ec4608a1f76453166bb312b2968aeaf4,2017-11-06 19:58:31
3,4,-3.022887,f52c8058817a332008a3c374318bf70c,4,Insatisfação com a descrição do produto.,furniture_decor,29c131c43e662b738339aafcc0ba00fc,3df020b72d3d44b3af9d110fa3940b65,2017-08-17 21:15:13
4,5,-3.836206,d69aa04a78d0651dba8163e736abecb5,1,Atendimento péssimo das lojas lannister tambem...,furniture_decor,e64a9b6ec4c7f4081be3a5f7ce77d6fe,63b464dbf392c7b80d12d932fa7cafed,2017-08-17 22:26:15


In [78]:
def build_evidence_context(results):

    evidence = []

    for _, row in results.iterrows():

        evidence.append(
            f"""Evidence {row['rank']}:
Review ID: {row['review_id']}
Rating: {row['review_score']}
Category: {row['category']}
Product ID: {row['product_id']}
Seller ID: {row['seller_id']}
Date: {row['order_purchase_timestamp']}
Review: {row['review_text']}"""
        )

    return "\n\n".join(evidence)

In [79]:
context = build_evidence_context(results)

print(context)

Evidence 1:
Review ID: 118cea1913e2f9d91f24066bf570fe20
Rating: 1
Category: furniture_decor
Product ID: 96ed3b4b62790bd962aac0b317109be4
Seller ID: cac4c8e7b1ca6252d8f20b2fc1a2e4af
Date: 2017-05-18 15:23:08
Review: péssima embalagem , tive que colar ele todo, fique com preguiça pelo preço de mandar de volta, mas não indico pra ninguém a loja...

Evidence 2:
Review ID: e728b7d132f8dfa2229cb6050e93a88f
Rating: 1
Category: furniture_decor
Product ID: e4e81849dd9ccdefe1e7398792a6a4b4
Seller ID: 77530e9772f57a62c906e1c21538ab82
Date: 2017-05-11 10:19:00
Review: Recebi a compra pela metade, comprei 03 kits e só veio 01... péssima compra... já formalizei a reclamação a loja e ate o momento nada de retorno!

Evidence 3:
Review ID: 574ff7b2058b5fbb967b36ccf690c472
Rating: 3
Category: furniture_decor
Product ID: 3c5ba90e09547db90868e803d139de30
Seller ID: ec4608a1f76453166bb312b2968aeaf4
Date: 2017-11-06 19:58:31
Review: Sou cliente stark há anos, mas o motivo do meu descontentamento é porque pe

In [82]:
print("Context length:", len(context))
print("Evidence blocks:", context.count("Evidence "))

Context length: 1689
Evidence blocks: 5


In [88]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.


In [91]:
import ollama

In [106]:
def build_why_prompt(query, context):
    return f"""
You are a business intelligence analyst.

Answer the user's question using ONLY the retrieved evidence below.

User question:
{query}

Retrieved evidence:
{context}

Rules:
- Only make claims directly supported by the retrieved review text.
- A review belongs under "Direct evidence" ONLY if its actual review text
  explicitly supports the user's question.
- If a review does not explicitly mention the requested issue, it MUST NOT
  appear under "Direct evidence", even if it is semantically related.
- If a review is only loosely related, put it under "Related themes" or ignore it.
- Do NOT assume correlation or causation unless the review explicitly states it.
- Do NOT generalize from a small sample to all customers.
- Cite Review IDs for every direct-evidence claim.
- Do not invent statistics, trends, causes, frequencies, or business conclusions.
- If fewer than 2 retrieved reviews directly support the question,
  explicitly state that direct evidence is limited.
- Never infer "dimensions or size" from "product description", "product quality",
  "delivery", "seller response", or "customer satisfaction".
- If the retrieved evidence does not adequately answer the question, say so.

Confidence must be:
- High = multiple strong pieces of direct evidence
- Medium = some direct evidence but limited coverage
- Low = only one direct match, weak evidence, or mostly indirect evidence

Return exactly:

Main issue:
<answer>

Direct evidence:
- <only reviews that directly support the question, with Review IDs>

Related themes:
- <only genuinely related issues>

Confidence:
High / Medium / Low

Caveats:
<limitations of the retrieved evidence>
"""

In [107]:
results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

print(response["message"]["content"])

Main issue:
Customers complaining about product dimensions or size

Direct evidence:
- Review ID: 95d9b32ceb14e27ec00467d1792b457e, Rating: 1
- Review ID: 3, Rating: 1
- Review ID: 1e39301e37ad6658793dfe56c6e0c077, Rating: 1

Related themes:
- Product quality issues (e.g., "O produto veio com dimensões erradas e avariado")
- Delivery issues (e.g., "produto atrasado sem resposta da loja")

Confidence:
Low

Caveats:
The retrieved evidence is limited to only three reviews that directly mention product dimensions or size as a complaint. These reviews do not provide a comprehensive understanding of the issue, and the confidence level is therefore low.


## Done with the prompt thing

In [125]:
generation_eval = pd.DataFrame({
    "query": [
        "customers complaining about damaged or defective products",
        "customers complaining about late delivery",
        "customers complaining about missing items",
        "customers complaining about poor product quality",
        "customers complaining about sellers not responding",
        "customers asking for refunds or replacements",
        "customers receiving the wrong product",
        "customers complaining about poor packaging",
        "customers saying their order was not delivered",
        "customers complaining about product dimensions or size"
    ],
    "direct_evidence_precision": [None] * 10,
    "citation_correctness": [None] * 10,
    "unsupported_claims": [None] * 10,
    "answer_relevance": [None] * 10,
    "evidence_sufficiency": [None] * 10
})

generation_eval

,query,direct_evidence_precision,citation_correctness,unsupported_claims,answer_relevance,evidence_sufficiency
0,customers complaining about damaged or defecti...,None,None,None,None,None
1,customers complaining about late delivery,None,None,None,None,None
2,customers complaining about missing items,None,None,None,None,None
3,customers complaining about poor product quality,None,None,None,None,None
4,customers complaining about sellers not respon...,None,None,None,None,None
5,customers asking for refunds or replacements,None,None,None,None,None
6,customers receiving the wrong product,None,None,None,None,None
7,customers complaining about poor packaging,None,None,None,None,None
8,customers saying their order was not delivered,None,None,None,None,None
9,customers complaining about product dimensions...,None,None,None,None,None


In [124]:
query = "customers complaining about product dimensions or size"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                           review_text
    1 f52c8058817a332008a3c374318bf70c             4                                                              Insatisfação com a descrição do produto.
    2 61fcfc9aad23649e8a5b653d539a2d19             1 deveria informar o tamanho e também apaga quando coloca ela no bafo portanto não recomendo o vendedor
    3 95d9b32ceb14e27ec00467d1792b457e             1                                                      O produto veio com dimensões erradas e avariado.
    4 1e39301e37ad6658793dfe56c6e0c077             1                                                                produto atrasado sem resposta da loja 
    5 25797d45a4e622c827ed6511a8e2aaf0             5                                                             Essa loja entregou o produto solicitado .

===== LLM ANSWER =====
Main issue:
Customers complaining about produc

In [126]:
generation_eval.loc[
    generation_eval["query"] == "customers complaining about product dimensions or size",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [1/3, 1/3, 1, 1.0, 0]

In [128]:
query = "customers complaining about damaged or defective products"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                                                                                                                                review_text
    1 574ff7b2058b5fbb967b36ccf690c472             3       Sou cliente stark há anos, mas o motivo do meu descontentamento é porque pela segunda vez me enviaram um produto danificado e esta entrega está atrasada. Ao tentar entrar em contato com a Central 
    2 5b2602b06d6360ecf96806983c564cb8             1 O produto veio com defeito, a cortina veio com a embalagem violada e o produto estava danificado, queimado.\r\nFiz a reclamação direto com a loja e eles ficaram de me retornar mas não retornaram até hoj
    3 f0b8a0026b6c510825eef1286f2b27a5             1                                                                                                          O produto comprado veio incompleto. Fiz a reclamação e até o momento não r

In [129]:
generation_eval.loc[
    generation_eval["query"] == "customers complaining about damaged or defective products",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [0.60, 0.60, 0, 1.0, 1.0]

In [131]:
query = "customers complaining about late delivery"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                                                                                                                          review_text
    1 574ff7b2058b5fbb967b36ccf690c472             3 Sou cliente stark há anos, mas o motivo do meu descontentamento é porque pela segunda vez me enviaram um produto danificado e esta entrega está atrasada. Ao tentar entrar em contato com a Central 
    2 f242ce44c6572a94907a446b8fda32c8             1                     Olá!\r\nMinha queixa é a não entrega na residência. Tive que buscar no correio. Tenho um produto que foi devolvido, e ainda nem sei o que fazer. Talvez um telefonema ajudasse. 
    3 d0fc966616272c040a351980b50a4aca             2                                                                                                                                                            estou desapontada com o atraso da entrega


In [132]:
generation_eval.loc[
    generation_eval["query"] == "customers complaining about late delivery",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [0.50, 0.50, 0, 1.0, 0]

In [134]:
query = "customers complaining about missing items"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                                                                                                                             review_text
    1 f0b8a0026b6c510825eef1286f2b27a5             1                                                                                                       O produto comprado veio incompleto. Fiz a reclamação e até o momento não recebi qualquer retorno.
    2 1e39301e37ad6658793dfe56c6e0c077             1                                                                                                                                                                  produto atrasado sem resposta da loja 
    3 7c48196c9f5a3ce913fbfb0076318f7c             5                                                                        Produto pedido estava esgotado. Porém a loja entrou em contato, foi super atenciosa e enviou outro produto de minha p

In [140]:
generation_eval.loc[
    generation_eval["query"] == "customers complaining about missing items",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [2/3, 2/3, 0, 1.0, 1.0]

In [135]:
query = "customers complaining about poor product quality"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                                                                    review_text
    1 afa4452e5e5b79681346303da60abbbd             2                                                                           Produto de péssima qualidade, muito diferente do oferecido no site. 
    2 2bf6e0c426c367739347388f4aef4dee             3                                                                                                     Qualidade do produto inferior ao desejado!
    3 1e39301e37ad6658793dfe56c6e0c077             1                                                                                                         produto atrasado sem resposta da loja 
    4 85107a1dcf6d0ecc4e46eba9f50a8efa             1 Produto ruim de má qualidade.\r\nLaterais da cortina desfiando / desmanchando.\r\nPara um produto de 165,00, não condiz .\r\nInsatisfeito.\r\n
    5 d69aa04a78d065

In [141]:
generation_eval.loc[
    generation_eval["query"] == "customers complaining about poor product quality",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [0.75, 0.75, 0, 1.0, 1.0]

In [136]:
query = "customers asking for refunds or replacements"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                                                                                                                                  review_text
    1 4c128092fecbd9c6dadcef8796cd1cbc             5                                  Recebi o pedido em menos de uma semana. Só que comprei com as medidas erradas e quero devolver o produto dentro do prazo do direito do consumidor e não estou conseguindo. 
    2 6704cd4e932a8a82840ce44ab5065d11             1             Desisti da compra enquanto o produto ainda estava na transportadora e a loja nao atende meu pedido. Nao respondem minha solicitacao de código de envio para devolução. Nao retornam meus emails.
    3 8cfd16cb42030ade9c35b2e614c62c9b             1 eu recebi o produto com defeito quero devolver e receber vale compra, ou o estorno do pagamento,\r\neu ja tenho outro produto no carrinho pra resolver esse problema , porem 

In [142]:
generation_eval.loc[
    generation_eval["query"] == "customers asking for refunds or replacements",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [0.75, 0.75, 0, 1.0, 1.0]

In [143]:
query = "customers receiving the wrong product"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                                                                                                                                  review_text
    1 574ff7b2058b5fbb967b36ccf690c472             3         Sou cliente stark há anos, mas o motivo do meu descontentamento é porque pela segunda vez me enviaram um produto danificado e esta entrega está atrasada. Ao tentar entrar em contato com a Central 
    2 8cfd16cb42030ade9c35b2e614c62c9b             1 eu recebi o produto com defeito quero devolver e receber vale compra, ou o estorno do pagamento,\r\neu ja tenho outro produto no carrinho pra resolver esse problema , porem não consigo um atendimento \r\n
    3 2d0a07007cc84621a2bc694505be4036             1                                                       Atraso na entrega sem dar satisfação ao cliente, que com certeza escolheu o produto com cuidado e com as melhores expec

In [144]:
generation_eval.loc[
    generation_eval["query"] == "customers receiving the wrong product",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [0.0, 0.0, 1, 0.5, 0]

In [145]:
query = "customers complaining about poor packaging"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                                                                                                                          review_text
    1 118cea1913e2f9d91f24066bf570fe20             1                                                                  péssima embalagem , tive que colar ele todo, fique com preguiça pelo preço de mandar de volta, mas não indico pra ninguém a loja...
    2 e728b7d132f8dfa2229cb6050e93a88f             1                                                    Recebi a compra pela metade, comprei 03 kits e só veio 01... péssima compra... já formalizei a reclamação a loja e ate o momento nada de retorno!
    3 574ff7b2058b5fbb967b36ccf690c472             3 Sou cliente stark há anos, mas o motivo do meu descontentamento é porque pela segunda vez me enviaram um produto danificado e esta entrega está atrasada. Ao tentar entrar em contato com a Central 


In [146]:
generation_eval.loc[
    generation_eval["query"] == "customers complaining about poor packaging",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [1/3, 1/3, 1, 1.0, 0]

In [147]:
query = "customers saying their order was not delivered"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                                                                                                                        review_text
    1 327d75c4d6dd076a7892dc910205f2e1             1                                                                                             O produto não foi entregue.\r\nMensagem da transportadora diz que "Objeto ainda não chegou à unidade" 
    2 1a77b1f3390930779939286b5d15c0ce             1 Recebi uma mensagem de que houve um problema na entrega. Consultei o site e vi que os correios acusavam o não recebimento do produto. Enviei uma mensagem à stark e à Loja que não me responderam.
    3 b7ce5a0a606b61681caee5b8cdd3fcbd             1                                                                                                                                         não chegou ainda a encomenda, o motivo deve ser o correio!
    4 10

In [148]:
generation_eval.loc[
    generation_eval["query"] == "customers saying their order was not delivered",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [1.0, 1.0, 1, 1.0, 0]

In [150]:
query = "customers complaining about sellers not responding"
category = "furniture_decor"

results = retrieve_reviews(query, category=category)
context = build_evidence_context(results)

prompt = build_why_prompt(query, context)

response = ollama.chat(
    model="llama3.2:3b",
    messages=[
        {"role": "user", "content": prompt}
    ]
)

retrieved_for_eval = results[
    ["rank", "review_id", "review_score", "review_text"]
].to_string(index=False)

llm_answer = response["message"]["content"]

print(retrieved_for_eval)
print("\n===== LLM ANSWER =====")
print(llm_answer)

 rank                        review_id  review_score                                                                                                                                                                                             review_text
    1 e74133b43b9b348ae8c6f3d419704ace             1                                    Fiz a compra de duas cortinas e só me enviaram uma abri uma reclamação e ate agora nem me responderam e nem deram satisfação é eu consumidora que estou no prejuízo.
    2 1e39301e37ad6658793dfe56c6e0c077             1                                                                                                                                                                  produto atrasado sem resposta da loja 
    3 98c303ce6a5deb174c7a4b76c882d940             1                                                                                                         Produto comprado e não entregue no prazo. O fornecedor não deu satisfação a respeito

In [151]:
generation_eval.loc[
    generation_eval["query"] == "customers complaining about sellers not responding",
    [
        "direct_evidence_precision",
        "citation_correctness",
        "unsupported_claims",
        "answer_relevance",
        "evidence_sufficiency"
    ]
] = [0.50, 0.50, 0, 1.0, 1.0]

In [152]:
generation_eval

,query,direct_evidence_precision,citation_correctness,unsupported_claims,answer_relevance,evidence_sufficiency
0,customers complaining about damaged or defecti...,0.6,0.6,0,1.0,1.0
1,customers complaining about late delivery,0.5,0.5,0,1.0,0
2,customers complaining about missing items,0.666667,0.666667,0,1.0,1.0
3,customers complaining about poor product quality,0.75,0.75,0,1.0,1.0
4,customers complaining about sellers not respon...,0.5,0.5,0,1.0,1.0
5,customers asking for refunds or replacements,0.75,0.75,0,1.0,1.0
6,customers receiving the wrong product,0.0,0.0,1,0.5,0
7,customers complaining about poor packaging,0.333333,0.333333,1,1.0,0
8,customers saying their order was not delivered,1.0,1.0,1,1.0,0
9,customers complaining about product dimensions...,0.333333,0.333333,1,1.0,0


In [153]:
print("Mean Direct Evidence Precision:",
      generation_eval["direct_evidence_precision"].mean())

print("Mean Citation Correctness:",
      generation_eval["citation_correctness"].mean())

print("Unsupported Claim Rate:",
      generation_eval["unsupported_claims"].mean())

print("Mean Answer Relevance:",
      generation_eval["answer_relevance"].mean())

print("Evidence Sufficiency Accuracy:",
      generation_eval["evidence_sufficiency"].mean())

Mean Direct Evidence Precision: 0.5433333333333332
Mean Citation Correctness: 0.5433333333333332
Unsupported Claim Rate: 0.4
Mean Answer Relevance: 0.95
Evidence Sufficiency Accuracy: 0.5


# Evidence-Validation Layer